In [1]:
from __future__ import annotations
from dataclasses import dataclass
from typing import Dict, List, Optional, Tuple
import os
import random
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.nn.utils.rnn import pad_sequence
from transformers import RobertaTokenizer

# ============================================================
# Repro (CPU-only)
# ============================================================
DEVICE = "cpu"

def set_seed(s: int = 42):
    random.seed(s); np.random.seed(s); torch.manual_seed(s)

# ============================================================
# Config
# ============================================================
@dataclass
class Config:
    seed: int = 42
    limit: int = 512
    max_in_len: int = 512
    max_out_len: int = 256
    demo_data: bool = False
    n_agents: int = 3
    model_dim: int = 384
    n_heads: int = 6
    n_layers_enc: int = 4
    n_layers_dec: int = 4
    max_len_cap: int = 1024
    router_temp: float = 1.0
    dyn_epochs: int = 4
    dyn_batch: int = 8
    dyn_lr: float = 2e-4
    lb_lambda: float = 0.25
    entropy_lambda: float = 0.10
    router_lambda: float = 0.0
    ft_epochs: int = 3
    ft_batch: int = 8
    ft_lr: float = 5e-4
    agent_idx: int = 0
    ft_unfreeze_adapters: bool = True
    ft_unfreeze_dec_norms: bool = True
    decode_max_len: int = 160
    out_dir: str = "preds_static_role"

CFG = Config()

# ============================================================
# Dataset (SWE-bench)
# ============================================================
try:
    from datasets import load_dataset
    HAVE_HF = True
except ImportError:
    HAVE_HF = False

class SWEText2PatchData:
    def __init__(self, *, split: str = "train", limit: Optional[int] = 512,
                 max_in_len: int = 512, max_out_len: int = 256, demo_data: bool = False):
        if not HAVE_HF:
            raise RuntimeError("Install `datasets`: pip install datasets")
        
        self.max_in_len = max_in_len
        self.max_out_len = max_out_len
        self.split = split
        self.tok = RobertaTokenizer.from_pretrained("Salesforce/codet5-small")
        self.pad_token_id = self.tok.pad_token_id
        self.bos_token_id = self.tok.bos_token_id
        self.eos_token_id = self.tok.eos_token_id

        if demo_data:
            print("[Data] DEMO synthetic dataset")
            rng = random.Random(0)
            self.full_data: List[Tuple[str, str, str]] = []
            n = int(limit or 512)
            for i in range(n):
                title = f"Issue {i}: Widget broken"
                body = f"Repro {i}: click→crash, trace={rng.randint(0,999)}"
                patch = f"diff --git a/app.py b/app.py\n+print('fix {i}')\n"
                self.full_data.append((f"demo-{i}", title + "\n" + body, patch))
        else:
            print("[Data] Load SWE-bench…")
            ds = load_dataset("princeton-nlp/SWE-bench", split="train")
            if limit is not None:
                ds = ds.select(range(min(limit, len(ds))))
            rows = list(ds)

            def build_input(ex: Dict) -> str:
                parts: List[str] = []
                if ex.get("title"): parts.append(str(ex["title"]))
                if ex.get("problem_statement"): parts.append(str(ex["problem_statement"]))
                if ex.get("hints_text"): parts.append("HINTS: " + str(ex["hints_text"]))
                meta = []
                if ex.get("repo"): meta.append(f"repo={ex['repo']}")
                if ex.get("base_commit"): meta.append(f"base={ex['base_commit']}")
                if meta: parts.append(" [" + ", ".join(meta) + "]")
                return "".join(parts)

            def pick_patch(ex: Dict) -> str:
                for key in ("patch", "base_patch", "model_patch", "test_patch"):
                    if key in ex and ex[key]: return str(ex[key])
                return ""

            self.full_data = [(str(ex.get("instance_id", "")), build_input(ex), pick_patch(ex))
                              for ex in rows if pick_patch(ex).strip()]
            print(f"[Data] Loaded {len(self.full_data)} supervised pairs")

        self._split_dataset()
        print(f"[Data] {len(self.data)} supervised pairs for split='{split}'")

    def _split_dataset(self):
        np.random.seed(42)
        indices = np.arange(len(self.full_data))
        np.random.shuffle(indices)
        split_idx = int(0.8 * len(self.full_data))
        if self.split == "train":
            self.data = [self.full_data[i] for i in indices[:split_idx]]
        elif self.split == "test":
            self.data = [self.full_data[i] for i in indices[split_idx:]]
        else:
            self.data = self.full_data

    def as_tensors(self) -> Tuple[List[str], torch.Tensor, torch.Tensor]:
        ids, xs, ys = [], [], []
        for iid, issue, patch in self.data:
            x = self.tok(issue, max_length=self.max_in_len, padding='max_length', truncation=True,
                         return_tensors='pt', add_special_tokens=False)['input_ids'][0]
            y = self.tok(patch, max_length=self.max_out_len, padding='max_length', truncation=True,
                         return_tensors='pt', add_special_tokens=True)['input_ids'][0]
            ids.append(iid)
            xs.append(x)
            ys.append(y)
        X = torch.stack(xs)
        Y = torch.stack(ys)
        print(f"[Info] Loaded {len(ids)} pairs | vocab={self.tok.vocab_size} | X={X.shape} Y={Y.shape}")
        return ids, X, Y

# ============================================================
# Model, Losses, Training, Decoding
# ============================================================
class Encoder(nn.Module):
    def __init__(self, vocab_size: int, model_dim: int = 512, n_heads: int = 8, n_layers: int = 6,
                 max_len: int = 1024, pad_token_id: int = 0):
        super().__init__()
        self.pad_token_id = pad_token_id
        self.tok_embedding = nn.Embedding(vocab_size, model_dim, padding_idx=pad_token_id)
        self.pos_embedding = nn.Parameter(torch.randn(1, max_len, model_dim) * 0.01)
        layer = nn.TransformerEncoderLayer(d_model=model_dim, nhead=n_heads, batch_first=True)
        self.encoder = nn.TransformerEncoder(layer, num_layers=n_layers)
    def forward(self, x: torch.Tensor):
        B, T = x.shape
        h = self.tok_embedding(x) + self.pos_embedding[:, :T, :]
        mask = (x == self.pad_token_id)
        mem = self.encoder(h, src_key_padding_mask=mask)
        cls = mem[:, 0, :]
        return mem, cls, mask

class Decoder(nn.Module):
    def __init__(self, vocab_size: int, model_dim: int = 512, n_heads: int = 8, n_layers: int = 6,
                 max_len: int = 1024, pad_token_id: int = 0, tok_embedding: Optional[nn.Embedding] = None):
        super().__init__()
        self.pad_token_id = pad_token_id
        self.tok_embedding = tok_embedding if tok_embedding is not None else nn.Embedding(vocab_size, model_dim, padding_idx=pad_token_id)
        self.pos_embedding = nn.Parameter(torch.randn(1, max_len, model_dim) * 0.01)
        layer = nn.TransformerDecoderLayer(d_model=model_dim, nhead=n_heads, batch_first=True)
        self.decoder = nn.TransformerDecoder(layer, num_layers=n_layers)
    def _subsequent_mask(self, L: int, device) -> torch.Tensor:
        return torch.triu(torch.ones(L, L, dtype=torch.bool, device=device), diagonal=1)
    def forward(self, y_in: torch.Tensor, memory: torch.Tensor, src_key_padding_mask: torch.Tensor) -> torch.Tensor:
        B, Lt = y_in.shape
        y_emb = self.tok_embedding(y_in) + self.pos_embedding[:, :Lt, :]
        tgt_key_padding_mask = (y_in == self.pad_token_id)
        tgt_mask = self._subsequent_mask(Lt, y_in.device)
        return self.decoder(y_emb, memory, tgt_mask=tgt_mask, tgt_key_padding_mask=tgt_key_padding_mask,
                           memory_key_padding_mask=src_key_padding_mask)

class Agent(nn.Module):
    def __init__(self, model_dim: int, vocab_size: int, adapter_dim: int = 96):
        super().__init__()
        self.adapter = nn.Sequential(
            nn.LayerNorm(model_dim),
            nn.Linear(model_dim, adapter_dim),
            nn.GELU(),
            nn.Linear(adapter_dim, model_dim),
        )
        self.router_head = nn.Linear(model_dim, vocab_size)
        self.role_head = nn.Linear(model_dim, vocab_size)
    def project(self, states: torch.Tensor, head: str = "router") -> torch.Tensor:
        h = self.adapter(states)
        layer = self.router_head if head == "router" else self.role_head
        return layer(h)

class RoutingNetwork(nn.Module):
    def __init__(self, model_dim: int, n_agents: int, temperature: float = 1.0):
        super().__init__()
        self.linear = nn.Linear(model_dim, n_agents)
        self.temperature = temperature
    def forward(self, cls_feat: torch.Tensor):
        logits = self.linear(cls_feat) / max(self.temperature, 1e-6)
        probs = torch.softmax(logits, dim=-1)
        return logits, probs

class AssignmentModule:
    def __init__(self, n_agents: int): self.n_agents = n_agents
    def __call__(self, user_id: int) -> int:
        if isinstance(user_id, torch.Tensor): return int((user_id % self.n_agents).item())
        return int(user_id) % self.n_agents

class DualRoutingModule(nn.Module):
    def __init__(self, model_dim: int, agents: nn.ModuleList, temperature: float = 1.0):
        super().__init__()
        self.agents = agents
        self.routing = RoutingNetwork(model_dim, n_agents=len(agents), temperature=temperature)
        self.assign = AssignmentModule(n_agents=len(agents))
    @property
    def temperature(self) -> float: return self.routing.temperature
    @temperature.setter
    def temperature(self, t: float): self.routing.temperature = t
    def forward(self, dec_states: torch.Tensor, cls: torch.Tensor, *,
                mode: str = "dynamic", user_id: Optional[int] = None,
                return_routing: bool = False) -> Tuple[torch.Tensor, Optional[torch.Tensor], Optional[torch.Tensor]]:
        if mode == "dynamic":
            rlogits, rprobs = self.routing(cls)
            pick = rprobs.argmax(dim=-1).tolist()
            outs = [self.agents[ai].project(dec_states[b:b+1], head="router") for b, ai in enumerate(pick)]
            outs = torch.cat(outs, dim=0)
            return (outs, rlogits, rprobs) if return_routing else (outs, None, None)
        elif mode == "static":
            assert user_id is not None, "user_id required for static routing"
            idx = self.assign(user_id)
            outs = self.agents[idx].project(dec_states, head="role")
            return (outs, None, None)
        else:
            raise ValueError("mode must be 'dynamic' or 'static'")

class AgenticTransformerSeq2Seq(nn.Module):
    def __init__(self, vocab_size: int, n_agents: int = 3, model_dim: int = 512, n_heads: int = 8,
                 n_layers_enc: int = 6, n_layers_dec: int = 6, max_len: int = 1024, pad_token_id: int = 0,
                 router_temperature: float = 1.0):
        super().__init__()
        self.encoder = Encoder(vocab_size, model_dim, n_heads, n_layers_enc, max_len, pad_token_id)
        self.decoder = Decoder(vocab_size, model_dim, n_heads, n_layers_dec, max_len, pad_token_id,
                               tok_embedding=self.encoder.tok_embedding)
        agents = nn.ModuleList([Agent(model_dim, vocab_size) for _ in range(n_agents)])
        self.dual = DualRoutingModule(model_dim, agents, temperature=router_temperature)
        self.pad_token_id = pad_token_id
    def encode(self, x: torch.Tensor): return self.encoder(x)
    def decode_states(self, y_in: torch.Tensor, memory: torch.Tensor, src_key_padding_mask: torch.Tensor):
        return self.decoder(y_in, memory, src_key_padding_mask)
    def forward_dynamic(self, x: torch.Tensor, y_in: torch.Tensor, return_routing: bool = False):
        memory, cls, src_mask = self.encode(x)
        dec_states = self.decode_states(y_in, memory, src_mask)
        return self.dual(dec_states, cls, mode="dynamic", return_routing=return_routing)
    def forward_static(self, x: torch.Tensor, y_in: torch.Tensor, *, user_id: int):
        memory, cls, src_mask = self.encode(x)
        dec_states = self.decode_states(y_in, memory, src_mask)
        logits, _, _ = self.dual(dec_states, cls, mode="static", user_id=user_id, return_routing=False)
        return logits

class SeqCELoss(nn.Module):
    def __init__(self, pad_token_id: int):
        super().__init__(); self.ce = nn.CrossEntropyLoss(ignore_index=pad_token_id)
    def forward(self, logits: torch.Tensor, targets: torch.Tensor) -> torch.Tensor:
        B, L, V = logits.shape
        return self.ce(logits.reshape(B*L, V), targets.reshape(B*L))

class RouterSupervisionLoss(nn.Module):
    def __init__(self): super().__init__(); self.ce = nn.CrossEntropyLoss()
    def forward(self, rlogits: torch.Tensor, agent_targets: torch.Tensor) -> torch.Tensor:
        return self.ce(rlogits, agent_targets)

def shift_targets(y: torch.Tensor) -> Tuple[torch.Tensor, torch.Tensor]:
    return y[:, :-1], y[:, 1:]

def loss_load_balance(probs: torch.Tensor) -> torch.Tensor:
    mean = probs.mean(dim=0); n = probs.size(1)
    return ((mean - 1.0 / n) ** 2).sum()

def loss_entropy(probs: torch.Tensor) -> torch.Tensor:
    return -torch.sum(probs * torch.log(probs + 1e-8), dim=1).mean()

@torch.no_grad()
def k_token_metrics(logits: torch.Tensor, targets: torch.Tensor, k: int, pad_token_id: int) -> Dict[str, float]:
    B, L, V = logits.shape
    k_eff = min(k, L)
    logits_k, targets_k = logits[:, :k_eff, :], targets[:, :k_eff]
    ce = nn.CrossEntropyLoss(ignore_index=pad_token_id)(logits_k.reshape(-1, V), targets_k.reshape(-1))
    preds = logits_k.argmax(dim=-1)
    token_acc = ((preds == targets_k) & (targets_k != pad_token_id)).float().sum() / ((targets_k != pad_token_id).float().sum() + 1e-8)
    em = ((preds == targets_k) | (targets_k == pad_token_id)).all(dim=1).float().mean()
    return {"ce_first_k": float(ce.item()), "token_acc_first_k": float(token_acc.item()), "em_first_k": float(em.item())}

@torch.no_grad()
def evaluate(model: AgenticTransformerSeq2Seq, X: torch.Tensor, Y: torch.Tensor, *,
             mode: str = "static", user_id: int = 0, k: int = 32, device: str = DEVICE) -> Dict[str, float]:
    model.to(device); model.eval()
    seq_ce = SeqCELoss(pad_token_id=model.pad_token_id)
    xb = X.to(device); yb = Y.to(device)
    y_in, y_tgt = shift_targets(yb)
    if mode == "dynamic":
        logits, *_ = model.forward_dynamic(xb, y_in, return_routing=True)
    else:
        logits = model.forward_static(xb, y_in, user_id=user_id)
    full_ce = seq_ce(logits, y_tgt).item()
    preds = logits.argmax(dim=-1)
    token_acc = ((preds == y_tgt) & (y_tgt != model.pad_token_id)).float().sum() / ((y_tgt != model.pad_token_id).float().sum() + 1e-8)
    firstk = k_token_metrics(logits, y_tgt, k=k, pad_token_id=model.pad_token_id)
    return {"seq_ce": full_ce, "token_acc": float(token_acc.item()), **{f"first{k}_"+k_: v for k_, v in firstk.items()}}

def train_dynamic(model: AgenticTransformerSeq2Seq, X: torch.Tensor, Y: torch.Tensor, *,
                  epochs: int = 4, batch_size: int = 8, lr: float = 2e-4,
                  lb_lambda: float = 0.25, entropy_lambda: float = 0.10,
                  router_targets: Optional[torch.Tensor] = None, router_lambda: float = 0.0,
                  device: str = DEVICE, k_report: int = 32) -> None:
    model.to(device)
    opt = optim.Adam(model.parameters(), lr=lr)
    seq_ce = SeqCELoss(pad_token_id=model.pad_token_id)
    router_sup = RouterSupervisionLoss() if (router_targets is not None and router_lambda > 0.0) else None
    N = X.size(0)
    for ep in range(1, epochs + 1):
        model.train(); tot_ce = tot_lb = tot_ent = tot_r = 0.0
        for i in range(0, N, batch_size):
            xb = X[i:i+batch_size].to(device); yb = Y[i:i+batch_size].to(device)
            y_in, y_tgt = shift_targets(yb)
            logits, rlogits, rprobs = model.forward_dynamic(xb, y_in, return_routing=True)
            loss_ce = seq_ce(logits, y_tgt)
            lbl = loss_load_balance(rprobs); ent = loss_entropy(rprobs)
            loss = loss_ce + lb_lambda * lbl + entropy_lambda * ent
            if router_sup is not None:
                tgt_agents = router_targets[i:i+batch_size].to(device)
                rloss = router_sup(rlogits, tgt_agents)
                loss = loss + router_lambda * rloss
                tot_r += float(rloss.detach()) * xb.size(0)
            opt.zero_grad(); loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step()
            B = xb.size(0)
            tot_ce += float(loss_ce.detach())*B; tot_lb += float(lbl.detach())*B; tot_ent += float(ent.detach())*B
        print(f"[Dynamic][Ep {ep}] CE {tot_ce/float(N):.3f} | LB {tot_lb/float(N):.3f} | Ent {tot_ent/float(N):.3f}"
              + (f" | Router {tot_r/float(N):.3f}" if router_sup is not None else ""))
        xb = X[:batch_size].to(device); yb = Y[:batch_size].to(device)
        y_in, y_tgt = shift_targets(yb)
        logits, *_ = model.forward_dynamic(xb, y_in, return_routing=True)
        m = k_token_metrics(logits, y_tgt, k=k_report, pad_token_id=model.pad_token_id)
        print(f"[Dynamic][Ep {ep}] first-{k_report}:", m)

def _set_ft_requires_grad(model: AgenticTransformerSeq2Seq, *, user_id: int,
                          unfreeze_adapters: bool, unfreeze_dec_norms: bool):
    for p in model.parameters():
        p.requires_grad = False
    if unfreeze_dec_norms:
        for name, p in model.decoder.named_parameters():
            if "norm" in name:
                p.requires_grad = True
    idx = user_id % len(model.dual.agents)
    ag = model.dual.agents[idx]
    for name, p in ag.named_parameters():
        if name.startswith("role_head"):
            p.requires_grad = True
        elif unfreeze_adapters and name.startswith("adapter"):
            p.requires_grad = True

def fine_tune_static(model: AgenticTransformerSeq2Seq, X: torch.Tensor, Y: torch.Tensor, *,
                     user_id: int, epochs: int = 3, batch_size: int = 8, lr: float = 5e-4,
                     unfreeze_adapters: bool = True, unfreeze_dec_norms: bool = True,
                     idxs: Optional[torch.Tensor] = None,
                     device: str = DEVICE):
    model.to(device)
    _set_ft_requires_grad(model, user_id=user_id,
                          unfreeze_adapters=unfreeze_adapters,
                          unfreeze_dec_norms=unfreeze_dec_norms)
    params = [p for p in model.parameters() if p.requires_grad]
    opt = optim.Adam(params, lr=lr)
    loss_fn = SeqCELoss(pad_token_id=model.pad_token_id)
    
    if idxs is None:
        X_ft, Y_ft = X, Y
    else:
        X_ft, Y_ft = X[idxs], Y[idxs]

    N = X_ft.size(0)                        
    if N == 0:
        print(f"[Static-FT][Agent {user_id}] No samples; skipping.")
        return
        
    for ep in range(1, epochs + 1):
        model.train(); total = 0.0
        for i in range(0, N, batch_size):
            xb = X_ft[i:i+batch_size].to(device); yb = Y_ft[i:i+batch_size].to(device)
            y_in, y_tgt = shift_targets(yb)
            logits = model.forward_static(xb, y_in, user_id=user_id)
            loss = loss_fn(logits, y_tgt)
            opt.zero_grad(); loss.backward()
            nn.utils.clip_grad_norm_(params, 1.0)
            opt.step()
            total += float(loss.detach()) * xb.size(0)
        print(f"[Static-FT][Agent {user_id}] Ep {ep}/{epochs} | SeqCE {total/float(N):.3f}")

@torch.no_grad()
def step_decode_logits(model: AgenticTransformerSeq2Seq, dec_states: torch.Tensor, cls: torch.Tensor,
                       user_id: Optional[int], mode: str) -> torch.Tensor:
    last = dec_states[:, -1:]
    if mode == "dynamic":
        rlogits, rprobs = model.dual.routing(cls)
        pick = rprobs.argmax(dim=-1).tolist()
        outs = [model.dual.agents[ai].project(last[b:b+1], head="router") for b, ai in enumerate(pick)]
        return torch.cat(outs, dim=0)
    else:
        assert user_id is not None
        idx = model.dual.assign(user_id)
        return model.dual.agents[idx].project(last, head="role")

@torch.no_grad()
def generate(model: AgenticTransformerSeq2Seq, X: torch.Tensor, *, max_len: int = 160,
             mode: str = "static", user_id: int = 0, top_k: Optional[int] = 50,
             top_p: Optional[float] = None) -> torch.Tensor:
    model.eval()
    memory, cls, src_mask = model.encode(X)
    B = X.size(0)
    ys = torch.full((B, 1), model.tok.bos_token_id, dtype=torch.long, device=X.device)
    for _t in range(1, max_len):
        dec = model.decode_states(ys, memory, src_mask)
        step_logits = step_decode_logits(model, dec, cls, user_id, mode).squeeze(1)
        logits = step_logits
        if top_k is not None:
            kth = torch.topk(logits, k=min(top_k, logits.size(-1)), dim=-1)
            mask = torch.full_like(logits, float("-inf"))
            mask.scatter_(1, kth.indices, kth.values); logits = mask
        if top_p is not None:
            probs = torch.softmax(logits, dim=-1)
            sorted_probs, sorted_idx = torch.sort(probs, descending=True, dim=-1)
            cum = torch.cumsum(sorted_probs, dim=-1)
            mask = cum > top_p
            mask[..., 1:] = mask[..., :-1].clone(); mask[..., 0] = False
            logits.scatter_(1, sorted_idx[mask], float("-inf"))
        next_tok = torch.distributions.Categorical(logits=logits).sample().unsqueeze(1)
        ys = torch.cat([ys, next_tok], dim=1)
        if (next_tok == model.tok.eos_token_id).all(): break
    return ys

@torch.no_grad()
def dump_patches_for_harness(model: AgenticTransformerSeq2Seq, tokenizer: RobertaTokenizer,
                             ids: List[str], X: torch.Tensor, out_dir: str, *,
                             mode: str = "static", user_id: int = 0, max_len: int = 160) -> None:
    os.makedirs(out_dir, exist_ok=True)
    preds = generate(model, X.to(DEVICE), max_len=max_len, mode=mode, user_id=user_id, top_k=None, top_p=None)
    for i, iid in enumerate(ids):
        seq = [int(t) for t in preds[i].tolist() if int(t) not in (tokenizer.pad_token_id, tokenizer.bos_token_id, tokenizer.eos_token_id)]
        text = tokenizer.decode(seq, skip_special_tokens=True)
        with open(os.path.join(out_dir, f"{iid}.pred.patch"), "w", encoding="utf-8") as f:
            f.write(text)

def run_all(cfg: Config = CFG):
    set_seed(cfg.seed)
    train_data = SWEText2PatchData(split="train", limit=cfg.limit, max_in_len=cfg.max_in_len,
                                   max_out_len=cfg.max_out_len, demo_data=cfg.demo_data)
    test_data = SWEText2PatchData(split="test", limit=cfg.limit, max_in_len=cfg.max_in_len,
                                  max_out_len=cfg.max_out_len, demo_data=cfg.demo_data)
    train_ids, X_train, Y_train = train_data.as_tensors()
    test_ids, X_test, Y_test = test_data.as_tensors()
    vocab_size = train_data.tok.vocab_size

    N_train = X_train.size(0)
    row_idx = torch.arange(N_train)
    agent_map = row_idx % cfg.n_agents
    ft_idxs = (agent_map == cfg.agent_idx).nonzero(as_tuple=False).squeeze(1)

    model = AgenticTransformerSeq2Seq(
        vocab_size=vocab_size, router_temperature=cfg.router_temp,
        n_agents=cfg.n_agents, model_dim=cfg.model_dim, n_heads=cfg.n_heads,
        n_layers_enc=cfg.n_layers_enc, n_layers_dec=cfg.n_layers_dec,
        max_len=max(cfg.max_len_cap, X_train.size(1)), pad_token_id=train_data.tok.pad_token_id
    )
    model.tok = train_data.tok
    print("[Stage 1] Dynamic training (global)")
    train_dynamic(model, X_train, Y_train,
                  epochs=cfg.dyn_epochs, batch_size=cfg.dyn_batch, lr=cfg.dyn_lr,
                  lb_lambda=cfg.lb_lambda, entropy_lambda=cfg.entropy_lambda,
                  router_targets=None, router_lambda=cfg.router_lambda,
                  device=DEVICE, k_report=32)
    print("Eval (dynamic):", evaluate(model, X_test, Y_test, mode="dynamic", k=32))

    print(f"[Stage 2] Static FT for agent {cfg.agent_idx}")
    if ft_idxs.numel() == 0:
        print(f"[Stage 2] No train samples for agent {cfg.agent_idx}; skipping static FT.")
    else:
        fine_tune_static(model, X_train, Y_train,
                         user_id=cfg.agent_idx, epochs=cfg.ft_epochs, batch_size=cfg.ft_batch, lr=cfg.ft_lr,
                         unfreeze_adapters=cfg.ft_unfreeze_adapters, unfreeze_dec_norms=cfg.ft_unfreeze_dec_norms,
                         idxs=ft_idxs, device=DEVICE)
    
    print("Eval (static/role) AFTER FT:", evaluate(model, X_test, Y_test, mode="static", user_id=cfg.agent_idx, k=32))
    dump_patches_for_harness(model, test_data.tok, test_ids, X_test, out_dir=cfg.out_dir,
                             mode="static", user_id=cfg.agent_idx, max_len=cfg.decode_max_len)
    print(f"[Dump] Wrote predictions to: {os.path.abspath(cfg.out_dir)}")
    try:
        print("Sample files:", sorted(os.listdir(cfg.out_dir))[:5])
    except Exception:
        pass
    return model, train_data, (test_ids, X_test, Y_test)

if __name__ == "__main__":
    model, data, tensors = run_all(CFG)

[Data] Load SWE-bench…
[Data] Loaded 256 supervised pairs
[Data] 204 supervised pairs for split='train'
[Data] Load SWE-bench…
[Data] Loaded 256 supervised pairs
[Data] 52 supervised pairs for split='test'
[Info] Loaded 204 pairs | vocab=32100 | X=torch.Size([204, 512]) Y=torch.Size([204, 256])
[Info] Loaded 52 pairs | vocab=32100 | X=torch.Size([52, 512]) Y=torch.Size([52, 256])
[Stage 1] Dynamic training (global)


KeyboardInterrupt: 